# 07a — Decision Framework

## The One-Sentence Version
The real skill isn't knowing the methods — it's knowing when to use each one.

## What You'll Build Intuition For
- A decision flowchart for choosing the right method
- How to apply it to realistic scenarios
- Common mistakes that trip up practitioners
- The meta-principle: start simple, escalate only if needed

## Prerequisites
- All of Modules 01–06 — this notebook synthesises everything

## The Story

You now know a dozen methods for dealing with high-dimensional data. PCA, t-SNE, UMAP, Lasso, Random Forests, autoencoders, VAEs... The danger is tool-itis: reaching for a method because it's cool rather than because it's right.

This notebook is your cheat sheet. A flowchart for choosing, worked examples for calibration, and a list of the mistakes that catch even experienced practitioners.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from utils.plotting import apply_style, COLOURS, PALETTE

apply_style()

## The Flowchart

When you have high-dimensional data and need to reduce it, start here.

In [ ]:
# Build the decision flowchart as a visual diagram
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_xticks([])
ax.set_yticks([])

def draw_box(ax, x, y, text, colour, width=2.2, height=0.6):
    rect = mpatches.FancyBboxPatch((x - width/2, y - height/2), width, height,
                                     boxstyle="round,pad=0.1", facecolor=colour,
                                     alpha=0.8, edgecolor="white", linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=8, fontweight="bold",
            wrap=True)

def draw_diamond(ax, x, y, text, colour):
    diamond = mpatches.FancyBboxPatch((x - 1.3, y - 0.4), 2.6, 0.8,
                                       boxstyle="round,pad=0.1", facecolor=colour,
                                       alpha=0.8, edgecolor="white", linewidth=2)
    ax.add_patch(diamond)
    ax.text(x, y, text, ha="center", va="center", fontsize=8, fontweight="bold")

# Root question
draw_diamond(ax, 5, 9.2, "Need interpretable features?", COLOURS["accent"])

# Yes branch: Feature Selection
ax.annotate("", xy=(2.5, 8.0), xytext=(4.0, 8.8),
            arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1.5))
ax.text(3.0, 8.5, "Yes", fontsize=9, color=COLOURS["signal"])

draw_box(ax, 2.5, 7.8, "Feature Selection", COLOURS["signal"])
draw_box(ax, 1.2, 6.8, "Linear?\n→ Lasso", "#93C5FD")
draw_box(ax, 2.5, 6.8, "Nonlinear?\n→ MI / Trees", "#93C5FD")
draw_box(ax, 3.8, 6.8, "Small data?\n→ RFE", "#93C5FD")

for x in [1.2, 2.5, 3.8]:
    ax.annotate("", xy=(x, 7.15), xytext=(2.5, 7.45),
                arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1))

# No branch: Transformation
ax.annotate("", xy=(7.5, 8.0), xytext=(6.0, 8.8),
            arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1.5))
ax.text(7.0, 8.5, "No", fontsize=9, color=COLOURS["noise"])

draw_box(ax, 7.5, 7.8, "Transformation", COLOURS["noise"])

# Try PCA first
ax.annotate("", xy=(7.5, 7.0), xytext=(7.5, 7.45),
            arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1.5))
draw_diamond(ax, 7.5, 6.6, "Try PCA first → clean elbow?", COLOURS["accent"])

# Yes: done
ax.annotate("", xy=(5.5, 5.7), xytext=(6.5, 6.2),
            arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1.5))
ax.text(5.8, 6.1, "Yes", fontsize=9, color=COLOURS["signal"])
draw_box(ax, 5.5, 5.5, "Done.\nUse PCA.", COLOURS["success"])

# No: nonlinear options
ax.annotate("", xy=(8.5, 5.5), xytext=(8.5, 6.2),
            arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1.5))
ax.text(8.7, 5.9, "No", fontsize=9, color=COLOURS["noise"])
draw_box(ax, 8.5, 5.2, "Data may be\nnonlinear", COLOURS["accent"])

draw_box(ax, 7.0, 4.2, "Visualisation\n→ UMAP", "#FCA5A5")
draw_box(ax, 8.5, 4.2, "Preprocessing\n→ Autoencoder", "#FCA5A5")
draw_box(ax, 10.0, 4.2, "Generation\n→ VAE", "#FCA5A5", width=1.6)

for x in [7.0, 8.5, 10.0]:
    ax.annotate("", xy=(x, 4.55), xytext=(8.5, 4.85),
                arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"], lw=1))

# Meta-principle
rect = mpatches.FancyBboxPatch((1.5, 1.0), 7.0, 1.8,
                                 boxstyle="round,pad=0.2", facecolor="#F3F4F6",
                                 alpha=0.9, edgecolor=COLOURS["neutral"], linewidth=2)
ax.add_patch(rect)
ax.text(5, 2.3, "The Meta-Principle", ha="center", fontsize=12, fontweight="bold")
ax.text(5, 1.7, "1. Start with PCA  →  2. Check explained variance  →  3. Escalate only if needed",
        ha="center", fontsize=10)
ax.text(5, 1.2, "Always verify: does the reduction improve downstream performance?",
        ha="center", fontsize=9, style="italic", color=COLOURS["neutral"])

ax.set_title("Decision Framework: Choosing the Right Method", fontsize=14)
plt.tight_layout()
plt.savefig("visuals/07a_flowchart.png", dpi=150, bbox_inches="tight")
plt.show()

## Scenario Testing

Let's walk through realistic scenarios using the flowchart.

In [ ]:
scenarios = [
    {
        "title": "500 patients, 20 lab values, predict mortality",
        "interpretable": True,
        "recommendation": "Lasso or RFE",
        "reasoning": "Clinicians need to know WHICH labs matter. Use Lasso for "
                     "automatic selection, or RFE with logistic regression. "
                     "Dataset is small enough for wrapper methods."
    },
    {
        "title": "50,000 gene expression levels, explore structure",
        "interpretable": False,
        "recommendation": "PCA first, then UMAP",
        "reasoning": "Exploratory — interpretability isn't the primary goal. "
                     "PCA for initial overview and explained variance curve. "
                     "UMAP for detailed cluster visualisation. Dataset is large, "
                     "so t-SNE would be too slow."
    },
    {
        "title": "100 simulation parameters, sensitivity analysis",
        "interpretable": True,
        "recommendation": "Sobol indices (≈ MI) + Lasso",
        "reasoning": "Need to know WHICH parameters drive the output. "
                     "Sobol/MI for nonlinear sensitivity, Lasso for linear. "
                     "Reduce to 5-10 influential parameters, then explore that subspace."
    },
    {
        "title": "Customer data, 200 features, explain to business",
        "interpretable": True,
        "recommendation": "Random Forest importance → top 10",
        "reasoning": "Business stakeholders need named features they can act on. "
                     "Tree importance is intuitive: 'purchase frequency is the #1 predictor.' "
                     "PCA components would be opaque to stakeholders."
    },
    {
        "title": "Image classification, 784 pixels",
        "interpretable": False,
        "recommendation": "Autoencoder or CNN features",
        "reasoning": "No one cares which pixel matters. Transformation is fine. "
                     "Try PCA first — if explained variance curve has a sharp elbow, PCA suffices. "
                     "Otherwise, autoencoder captures spatial nonlinear structure better."
    },
]

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, len(scenarios) + 0.5)
ax.set_xticks([])
ax.set_yticks([])

for i, s in enumerate(scenarios):
    y = len(scenarios) - i - 0.3
    colour = COLOURS["signal"] if s["interpretable"] else COLOURS["accent"]
    tag = "SELECT" if s["interpretable"] else "TRANSFORM"

    ax.text(0.1, y, f"{i+1}. {s['title']}", fontsize=10, fontweight="bold", va="top")
    ax.text(0.3, y - 0.35, f"→ {s['recommendation']}", fontsize=10, color=colour,
            fontweight="bold", va="top")
    ax.text(0.3, y - 0.6, s["reasoning"], fontsize=8, color=COLOURS["neutral"],
            va="top", wrap=True)
    ax.axhline(y=y - 0.85, color=COLOURS["grid"], linewidth=0.5)

ax.set_title("Scenario Testing: Walking Through the Flowchart", fontsize=14)
plt.tight_layout()
plt.savefig("visuals/07a_scenarios.png", dpi=150, bbox_inches="tight")
plt.show()

## Common Mistakes

These are the errors that trip up even experienced practitioners.

In [ ]:
mistakes = [
    ("Using t-SNE/UMAP output as model input",
     "These are visualisation tools, not preprocessing steps. The coordinates are "
     "optimisation artefacts — they don't carry meaningful numeric relationships.",
     "Use PCA or autoencoders for preprocessing."),

    ("Using PCA on clearly nonlinear data",
     "If your data lives on a curved manifold (Swiss roll, clusters with nonlinear "
     "boundaries), PCA will project through the structure instead of following it.",
     "Check PCA reconstruction error. If it's high, try nonlinear methods."),

    ("Over-reducing: too few components",
     "Aggressively reducing to 2D 'for visualisation' and then using those 2 "
     "components for modelling. You lose too much information.",
     "Choose k based on explained variance (95% threshold) or downstream task performance."),

    ("Under-reducing: keeping noise 'just in case'",
     "Keeping 100 components when 10 capture 99% of variance. The extra 90 are "
     "pure noise that will degrade your model.",
     "Trust the explained variance curve. Drop dimensions that contribute nothing."),

    ("Not standardising before PCA",
     "PCA finds directions of maximum variance. If one feature is in metres and "
     "another in millimetres, the mm feature dominates PC1 just because of units.",
     "Always StandardScaler before PCA (unless features are already comparable)."),
]

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 10)
ax.set_ylim(0, len(mistakes) + 0.5)
ax.set_xticks([])
ax.set_yticks([])

for i, (title, problem, fix) in enumerate(mistakes):
    y = len(mistakes) - i - 0.2
    ax.text(0.1, y, f"{i+1}. {title}", fontsize=10, fontweight="bold",
            color=COLOURS["noise"], va="top")
    ax.text(0.3, y - 0.3, f"Problem: {problem}", fontsize=8,
            color=COLOURS["neutral"], va="top", wrap=True)
    ax.text(0.3, y - 0.6, f"Fix: {fix}", fontsize=8,
            color=COLOURS["signal"], va="top", fontweight="bold", wrap=True)
    ax.axhline(y=y - 0.85, color=COLOURS["grid"], linewidth=0.5)

ax.set_title("Common Mistakes in Dimensionality Reduction", fontsize=14)
plt.tight_layout()
plt.savefig("visuals/07a_mistakes.png", dpi=150, bbox_inches="tight")
plt.show()

## The Meta-Principle

If you remember one thing from this entire course:

> **Start simple. Escalate only if needed. Verify the result.**

1. **Start with PCA.** It's fast, deterministic, and often sufficient.
2. **Check the explained variance curve.** If 95% of variance is captured in 10 components out of 1000, you're done.
3. **Check downstream performance.** Did the reduction help your model? If accuracy dropped, you removed too much signal.
4. **Show a domain expert.** Does the 2D visualisation make sense? Do the selected features align with domain knowledge?
5. **Escalate only if PCA clearly fails.** High reconstruction error, no clean elbow, poor downstream performance — *then* try UMAP, autoencoders, etc.

The goal is never to use the fanciest method. It's to find the simplest method that captures the structure in your data.

<details>
<summary><b>The Maths Behind This</b></summary>

The meta-principle is formalised by the **bias-variance trade-off** and **minimum description length (MDL)**.

**MDL principle:** The best model is the one that achieves the shortest total description: model complexity + data given the model.

$$\text{MDL}(M) = L(M) + L(D | M)$$

A simpler model (PCA with few components) has low $L(M)$ but may have high $L(D|M)$ if the data is nonlinear. A complex model (deep autoencoder) has high $L(M)$ but can achieve low $L(D|M)$. The optimal model balances both.

**In practice:** PCA has $O(dk)$ parameters. A deep autoencoder has $O(d \times h + h \times k)$ or more. The autoencoder's extra capacity is only justified when the data has enough nonlinear structure to pay for it.

</details>

## Where This Matters

**Every data science project.** The decision framework isn't academic — it's what separates a junior analyst (who reaches for the latest deep learning paper) from a senior one (who starts with PCA, checks if it works, and only escalates when the data demands it). The best practitioners are the ones who choose the simplest tool that works.

**Technical leadership.** When reviewing someone else's dimensionality reduction pipeline, the first questions should be: "Did you try PCA first?" and "What does the explained variance curve look like?" If those haven't been answered, nothing else matters.

## Feynman Check

1. **Walk through the flowchart for your own dataset or project.**

2. **Name three common mistakes people make with dimensionality reduction.**

---

Now that you know how to choose methods, let's apply dimensionality thinking to a domain where it's not always obvious: simulation parameter spaces. **07b — Simulation Parameter Spaces**.